<a href="https://colab.research.google.com/github/rishikacm2210-alt/FAI-Lab-36/blob/main/Building%20a%20next%20word%20prediction%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install tokenizers matplotlib tqdm

^C


In [ ]:
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from tqdm import tqdm

from tokenizers import ByteLevelBPETokenizer
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu
cpu


In [ ]:
with open("tiny shakespeare dataset.txt","r",encoding="utf-8") as f:
    text = f.read()

print("Characters :",len(text))
print(text[:500])

Characters : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
Characters : 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citi

In [ ]:
train_end = int(0.9 * len(text))
valid_end = int(0.95 * len(text))

train_text = text[:train_end]
valid_text = text[train_end:valid_end]
test_text = text[valid_end:]

print(len(train_text))
print(len(valid_text))
print(len(test_text))

1003854
55770
55770
1003854
55770
55770


In [ ]:
Path("data").mkdir(exist_ok=True)

with open("data/train.txt","w",encoding="utf-8") as f:
    f.write(train_text)

with open("data/valid.txt","w",encoding="utf-8") as f:
    f.write(valid_text)

with open("data/test.txt","w",encoding="utf-8") as f:
    f.write(test_text)

print("Done")

Done
Done


In [ ]:
tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=["data/train.txt"],
    vocab_size=8000,
    min_frequency=2,
    special_tokens=[
        "<pad>",
        "<bos>",
        "<eos>",
        "<unk>"
    ]
)

Path("tokenizer").mkdir(exist_ok=True)

tokenizer.save_model("tokenizer")

['tokenizer/vocab.json', 'tokenizer/merges.txt']

['tokenizer/vocab.json', 'tokenizer/merges.txt']

In [ ]:
tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)

VOCAB_SIZE = tokenizer.get_vocab_size()

print(VOCAB_SIZE)

8000
8000


In [ ]:
train_tokens = tokenizer.encode(train_text).ids
valid_tokens = tokenizer.encode(valid_text).ids
test_tokens = tokenizer.encode(test_text).ids

print(len(train_tokens))
print(len(valid_tokens))
print(len(test_tokens))

284546
17175
17897
284546
17175
17897


In [ ]:
BLOCK_SIZE = 128

BATCH_SIZE = 32

EMBED_DIM = 256

NUM_HEADS = 8

NUM_LAYERS = 6

FFN_DIM = 1024

DROPOUT = 0.1

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 0.01

EPOCHS = 10

In [ ]:
class LanguageDataset(Dataset):

    def __init__(self,tokens,block_size):

        self.tokens=tokens
        self.block_size=block_size

    def __len__(self):

        return len(self.tokens)-self.block_size

    def __getitem__(self,idx):

        x=torch.tensor(
            self.tokens[idx:idx+self.block_size],
            dtype=torch.long
        )

        y=torch.tensor(
            self.tokens[idx+1:idx+self.block_size+1],
            dtype=torch.long
        )

        return x,y

In [ ]:
train_dataset = LanguageDataset(train_tokens,BLOCK_SIZE)
valid_dataset = LanguageDataset(valid_tokens,BLOCK_SIZE)
test_dataset = LanguageDataset(test_tokens,BLOCK_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

print(len(train_loader))
print(len(valid_loader))
print(len(test_loader))

8888
532
555
8888
532
555


In [ ]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, dropout):

        super().__init__()

        assert embed_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = dropout

    def forward(self, x):

        B, T, C = x.shape

        qkv = self.qkv(x)

        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True
        )

        out = out.transpose(1,2).contiguous().view(B,T,C)

        return self.proj(out)

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, embed_dim, hidden_dim, dropout):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(embed_dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, embed_dim),

            nn.Dropout(dropout)

        )

    def forward(self, x):

        return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self,
                 embed_dim,
                 num_heads,
                 hidden_dim,
                 dropout):

        super().__init__()

        self.ln1 = nn.LayerNorm(embed_dim)

        self.attn = MultiHeadSelfAttention(
            embed_dim,
            num_heads,
            dropout
        )

        self.ln2 = nn.LayerNorm(embed_dim)

        self.ff = FeedForward(
            embed_dim,
            hidden_dim,
            dropout
        )

    def forward(self, x):

        x = x + self.attn(self.ln1(x))

        x = x + self.ff(self.ln2(x))

        return x

In [ ]:
class GPTModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBED_DIM
        )

        self.position_embedding = nn.Embedding(
            BLOCK_SIZE,
            EMBED_DIM
        )

        self.dropout = nn.Dropout(DROPOUT)

        self.blocks = nn.ModuleList(

            [
                TransformerBlock(
                    EMBED_DIM,
                    NUM_HEADS,
                    FFN_DIM,
                    DROPOUT
                )

                for _ in range(NUM_LAYERS)
            ]

        )

        self.ln = nn.LayerNorm(EMBED_DIM)

        self.head = nn.Linear(
            EMBED_DIM,
            VOCAB_SIZE,
            bias=False
        )

        # Weight tying
        self.head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(self, idx):

        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = self.token_embedding(idx)

        x = x + self.position_embedding(positions)

        x = self.dropout(x)

        for block in self.blocks:

            x = block(x)

        x = self.ln(x)

        logits = self.head(x)

        return logits

In [ ]:
model = GPTModel().to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters : 6,819,840
Trainable Parameters : 6,819,840
Total Parameters : 6,819,840
Trainable Parameters : 6,819,840


In [ ]:
sample_x, sample_y = next(iter(train_loader))

sample_x = sample_x.to(device)

with torch.no_grad():

    out = model(sample_x)

print("Input Shape :", sample_x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([32, 128])
Output Shape: torch.Size([32, 128, 8000])
Input Shape : torch.Size([32, 128])
Output Shape: torch.Size([32, 128, 8000])


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [ ]:
from torch.amp import autocast, GradScaler

scaler = GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0

    for x, y in tqdm(loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = model(x)

            loss = criterion(
                logits.view(-1, VOCAB_SIZE),
                y.view(-1)
            )

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
@torch.no_grad()

def validate(model, loader):

    model.eval()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        loss = criterion(
            logits.view(-1, VOCAB_SIZE),
            y.view(-1)
        )

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
best_loss = float("inf")

patience = 3

counter = 0

train_losses = []

valid_losses = []

perplexities = []

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    valid_loss = validate(
        model,
        valid_loader
    )

    scheduler.step()

    perplexity = math.exp(valid_loss)

    train_losses.append(train_loss)

    valid_losses.append(valid_loss)

    perplexities.append(perplexity)

    print(
        f"\nEpoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Validation Loss : {valid_loss:.4f}"
    )

    print(
        f"Perplexity : {perplexity:.2f}"
    )

    if valid_loss < best_loss:

        best_loss = valid_loss

        counter = 0

        torch.save(
            model.state_dict(),
            "best_gpt_model.pt"
        )

        print("Best model saved.")

    else:

        counter += 1

        if counter >= patience:

            print("Early stopping.")

            break

print("\nTraining Finished")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    valid_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training Curve")

plt.grid(True)

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    perplexities,
    marker="o"
)

plt.xlabel("Epoch")

plt.ylabel("Perplexity")

plt.title("Validation Perplexity")

plt.grid(True)

plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "best_gpt_model.pt",
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

In [ ]:
@torch.no_grad()
def predict_next_word(
    text,
    temperature=0.8,
    top_k=20
):

    model.eval()

    ids = tokenizer.encode(text).ids

    if len(ids) == 0:
        print("Please enter some text.")
        return None

    ids = ids[-BLOCK_SIZE:]

    x = torch.tensor([ids], dtype=torch.long).to(device)

    logits = model(x)

    logits = logits[:, -1, :] / temperature

    probs = F.softmax(logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, top_k)

    next_id = torch.multinomial(top_probs, 1)

    predicted_id = top_ids.gather(-1, next_id).item()

    predicted_word = tokenizer.decode([predicted_id])

    top5_probs, top5_ids = torch.topk(probs, 5)

    predictions = []

    for p, idx in zip(top5_probs[0], top5_ids[0]):

        predictions.append(
            (
                tokenizer.decode([idx.item()]),
                p.item()
            )
        )

    return predicted_word, predictions

In [ ]:
while True:

    sentence = input("\nEnter text (type 'exit' to quit): ")

    if sentence.lower() == "exit":
        break

    result = predict_next_word(sentence)

    if result is None:
        continue

    prediction, top5 = result

    print("\nPredicted Next Word")
    print("-------------------")
    print(prediction)

    print("\nTop 5 Predictions")

    for word, prob in top5:

        print(f"{word:<20} {prob*100:.2f}%")

In [ ]:
prediction, top5 = predict_next_word(
    "To be or not to"
)

print("Prediction :", prediction)

print()

for word, prob in top5:

    print(word, round(prob*100,2), "%")

In [ ]:
test_loss = validate(
    model,
    test_loader
)

print("Test Loss :", round(test_loss,4))

print("Test Perplexity :", round(math.exp(test_loss),2))

In [ ]:
parameters = sum(
    p.numel()
    for p in model.parameters()
)

size_mb = parameters * 4 / (1024**2)

print(f"Parameters : {parameters:,}")

print(f"Approx Model Size : {size_mb:.2f} MB")

print(f"Vocabulary Size : {VOCAB_SIZE}")

print(f"Context Length : {BLOCK_SIZE}")

In [ ]:
tokenizer.save_model("saved_tokenizer")

print("Tokenizer Saved Successfully")

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "vocab_size": VOCAB_SIZE
    },
    "next_word_prediction_model.pth"
)

print("Project Saved Successfully")